# The network by hand

For the default scenario (dry, new softs, full battery, 1 s behind on an 800 m
straight, Boost and Overtake Mode on) the trained network predicts a 44 %
chance of an overtake. This notebook recomputes that number from the saved
weights with nothing but multiplications, additions and one exponential,
one layer at a time, and checks the result against the library
(`overtake.predict`).

It then counts the arithmetic (Section 6.4 of the documentation) and shows how
training sets the weights, by computing one gradient by hand and checking it
against PyTorch.

PyTorch is used only to open the saved file and, at the end, to check the
gradient. Everything else is NumPy.

In [1]:
# !git clone https://github.com/SamGazz10/Gazzola3001.git
# %cd Gazzola3001
# !pip install -q -r requirements.txt

In [2]:
import os
import sys
from pathlib import Path

# Run from the repository root whether the notebook is opened from notebooks/ or the root.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import torch

from overtake.features import BINARY, CATEGORICAL, DEFAULTS, NUMERIC, VOCABS
from overtake.predict import DEFAULT_MODEL, OvertakePredictor

np.set_printoptions(precision=3, suppress=True, linewidth=110)

## 1. The scenario

The 19 inputs, at their default values:

In [3]:
scenario = dict(DEFAULTS)
pd.Series(scenario, name="value").to_frame()

,value
track_condition,dry
segment,straight
circuit,generic
attacker_tyre,soft
defender_tyre,medium
gap_s,1.0
run_length_m,800.0
attacker_battery,100.0
defender_battery,60.0
attacker_tyre_age,5.0


## 2. What is in the saved model file

`models/overtake_net.pt` holds five trained networks (an ensemble), the
training-set statistics used to scale the inputs, and the training history.
Each network is a set of named arrays of numbers:

* `embeddings.*`: one lookup table per kind of category (Section 4 below);
* `mlp.0`: first layer (weights and biases), followed in the model by SiLU
  (`mlp.1`) and dropout (`mlp.2`), which have no numbers of their own;
* `mlp.3`: second layer; `mlp.6`: output layer.

In [4]:
checkpoint = torch.load(DEFAULT_MODEL, weights_only=False)
nets = [{name: tensor.numpy() for name, tensor in state.items()} for state in checkpoint["model_states"]]
pre = checkpoint["preprocessor"]
assert not pre["physics_features"]  # the final model uses the raw inputs only

net = nets[0]
table = pd.DataFrame({"shape": {k: v.shape for k, v in net.items()},
                      "numbers": {k: v.size for k, v in net.items()}})
print(f"{len(nets)} networks, {table['numbers'].sum():,} numbers each, "
      f"{len(nets) * table['numbers'].sum():,} in total")
table

5 networks, 6,589 numbers each, 32,945 in total


,shape,numbers
embeddings.condition.weight,"(3, 2)",6
embeddings.segment.weight,"(4, 3)",12
embeddings.circuit.weight,"(25, 6)",150
embeddings.tyre.weight,"(5, 4)",20
mlp.0.weight,"(64, 33)",2112
mlp.0.bias,"(64,)",64
mlp.3.weight,"(64, 64)",4096
mlp.3.bias,"(64,)",64
mlp.6.weight,"(1, 64)",64
mlp.6.bias,"(1,)",1


## 3. Turning the scenario into numbers

The network only takes numbers, so each input is converted the same way it
was during training:

* **categories** (weather, segment, circuit, tyres) become their position in
  a fixed list (the vocabulary);
* **quantities** (gap, battery, ...) are standardised, $z = (x - \text{mean}) / \text{std}$,
  using the mean and standard deviation of the training set, so every input
  is on a similar scale (a gap of 1 s and a battery of 100 % become comparable
  numbers);
* **on/off flags** (Boost, Overtake Mode) become 1 or 0.

In [5]:
cat_index = {col: VOCABS[vocab].index(scenario[col]) for col, vocab in CATEGORICAL.items()}
pd.DataFrame({"value": {c: scenario[c] for c in CATEGORICAL},
              "vocabulary": {c: VOCABS[v] for c, v in CATEGORICAL.items()},
              "index": cat_index})

,value,vocabulary,index
track_condition,dry,"[dry, damp, wet]",0
segment,straight,"[straight, slow_corner, medium_corner, fast_co...",0
circuit,generic,"[generic, melbourne, shanghai, suzuka, bahrain...",0
attacker_tyre,soft,"[soft, medium, hard, intermediate, wet]",0
defender_tyre,medium,"[soft, medium, hard, intermediate, wet]",1


In [6]:
z = np.array([(scenario[c] - pre["means"][c]) / pre["stds"][c] for c in NUMERIC])
flags = np.array([float(scenario[c]) for c in BINARY])
pd.DataFrame({"value": [scenario[c] for c in NUMERIC],
              "training mean": [pre["means"][c] for c in NUMERIC],
              "training std": [pre["stds"][c] for c in NUMERIC],
              "standardised z": z}, index=NUMERIC).round(3)

,value,training mean,training std,standardised z
gap_s,1.0,1.025,0.563,-0.045
run_length_m,800.0,505.641,377.626,0.779
attacker_battery,100.0,57.446,32.046,1.328
defender_battery,60.0,57.572,32.027,0.076
attacker_tyre_age,5.0,19.997,11.835,-1.267
defender_tyre_age,15.0,20.036,11.841,-0.425
track_temp_c,35.0,32.597,11.845,0.203
attacker_skill,85.0,79.997,11.541,0.433
defender_skill,85.0,79.973,11.541,0.436
top_speed_delta_kph,0.0,0.002,4.980,-0.000


## 4. Embeddings: a learned lookup table for each category

An embedding is a table with one row per category value. Each row is a short
list of numbers learned during training, just like the weights. The circuit
table has 25 rows of 6 numbers: training placed circuits where overtaking is
similar close together in this 6-number space. "Looking up" a row is the same
as multiplying a one-hot vector (all zeros except a 1 at the index) by the
table, so an embedding is just a first layer for categorical inputs.

In [7]:
E_circuit = net["embeddings.circuit.weight"]
i = cat_index["circuit"]
one_hot = np.zeros(len(E_circuit))
one_hot[i] = 1.0
print(f"circuit table: {E_circuit.shape}, row {i} ({scenario['circuit']}):", E_circuit[i])
print("one-hot x table gives the same row:", np.allclose(one_hot @ E_circuit, E_circuit[i]))

circuit table: (25, 6), row 0 (generic): [ 0.177 -0.127  0.261 -0.104 -0.153 -0.081]
one-hot x table gives the same row: True


The network's input is the five looked-up rows (2 + 3 + 6 + 4 + 4 = 19
numbers; both tyre columns use the same tyre table), then the 11 standardised
quantities and the 3 flags: **33 numbers**.

In [8]:
def input_vector(net, cat_index, z, flags):
    rows = [net[f"embeddings.{vocab}.weight"][cat_index[col]] for col, vocab in CATEGORICAL.items()]
    return np.concatenate(rows + [z, flags])


x = input_vector(net, cat_index, z, flags)
print(x.shape)
print(x)

(33,)
[ 2.021 -0.031  0.129  0.818 -0.697  0.177 -0.127  0.261 -0.104 -0.153 -0.081 -1.419  0.159 -1.244 -1.006
 -1.571  0.717 -0.703  0.944 -0.045  0.779  1.328  0.076 -1.267 -0.425  0.203  0.433  0.436 -0.     0.002
  1.     1.     1.   ]


## 5. First hidden layer: 64 weighted sums, then SiLU

Each of the 64 units ("neurons") takes a weighted sum of all 33 inputs plus a
bias:

$$z^{(1)}_k = \sum_{j=1}^{33} W^{(1)}_{kj}\, x_j + b^{(1)}_k$$

All 64 sums at once are one matrix-vector product, $W^{(1)} x + b^{(1)}$, with
$W^{(1)}$ of size 64 × 33. Written out for unit 0 as a loop, and then for all
units as a matrix product:

In [9]:
W1, b1 = net["mlp.0.weight"], net["mlp.0.bias"]

total = 0.0
for j in range(33):
    total += W1[0, j] * x[j]
total += b1[0]

z1 = W1 @ x + b1
print(f"unit 0 by loop: {total:.6f}   by matrix product: {z1[0]:.6f}")
print("all 64:", z1.shape)

unit 0 by loop: -2.125289   by matrix product: -2.125289
all 64: (64,)


Then each sum goes through the **SiLU** activation,
$\text{SiLU}(z) = z \cdot \sigma(z)$ with $\sigma(z) = 1/(1 + e^{-z})$. It is
close to $z$ for positive $z$ and small for negative $z$ (never below
−0.28), a smooth version of "keep it if positive". This bend is what lets the network
represent curves and thresholds (such as the 1 s Overtake Mode cliff);
without it, stacked layers would collapse into one straight-line model, which
is logistic regression (15.7 points of error against the network's 2.0).

**Dropout** randomly zeroes 10 % of these values during training only, so
the network cannot rely on any single unit. At prediction it is switched
off, so there is nothing to compute.

In [10]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def silu(z):
    return z * sigmoid(z)


h1 = silu(z1)
print("first 8 sums:      ", z1[:8])
print("after SiLU:        ", h1[:8])
print(f"units with a positive sum, passing their signal on: {(z1 > 0).sum()} of 64 "
      f"(the rest come out small and slightly negative: SiLU never goes below {h1.min():.2f})")

first 8 sums:       [-2.125 -0.948 -0.041 -0.031 -0.674 -0.255 -2.074 -0.976]
after SiLU:         [-0.227 -0.265 -0.02  -0.015 -0.227 -0.111 -0.232 -0.267]
units with a positive sum, passing their signal on: 7 of 64 (the rest come out small and slightly negative: SiLU never goes below -0.28)


## 6. Second hidden layer and output

The second layer does the same with a 64 × 64 matrix. The output layer is a
single weighted sum of the 64 values: the **logit** (log-odds). The sigmoid
turns it into a probability between 0 and 1. The logit is a sum of 64
contributions $w_k h^{(2)}_k$ plus a bias; some push towards an overtake,
some against, and the prediction is the balance.

In [11]:
h2 = silu(net["mlp.3.weight"] @ h1 + net["mlp.3.bias"])
contributions = net["mlp.6.weight"][0] * h2
logit = contributions.sum() + net["mlp.6.bias"][0]
p = sigmoid(logit)
print("largest pushes for an overtake:", np.sort(contributions)[-3:][::-1])
print("largest pushes against:        ", np.sort(contributions)[:3])
print(f"sum of all 64 + bias {net['mlp.6.bias'][0]:.3f}: logit = {logit:.4f}")
print(f"P(overtake) = 1 / (1 + e^-logit) = {p:.4f}")

largest pushes for an overtake: [0.135 0.058 0.056]
largest pushes against:         [-0.111 -0.101 -0.076]
sum of all 64 + bias -0.157: logit = -0.3772
P(overtake) = 1 / (1 + e^-logit) = 0.4068


## 7. The ensemble: five networks, averaged

The five networks have the same shape but different numbers, because each
started training from a different random seed. Each has its own embedding
tables, so the input vector is rebuilt for each. The prediction is the mean
of their five probabilities.

In [12]:
def forward(net, cat_index, z, flags):
    x = input_vector(net, cat_index, z, flags)
    h1 = silu(net["mlp.0.weight"] @ x + net["mlp.0.bias"])
    h2 = silu(net["mlp.3.weight"] @ h1 + net["mlp.3.bias"])
    return sigmoid((net["mlp.6.weight"] @ h2 + net["mlp.6.bias"])[0])


members = [forward(n, cat_index, z, flags) for n in nets]
by_hand = float(np.mean(members))
library = OvertakePredictor.load().predict(scenario)

print("five networks:", np.round(members, 4))
print(f"by hand: {by_hand:.6f}   library: {library:.6f}   difference: {abs(by_hand - library):.1e}")
assert abs(by_hand - library) < 1e-5

five networks: [0.407 0.456 0.461 0.419 0.456]
by hand: 0.439775   library: 0.439775   difference: 6.0e-08


The members disagree by a few points; averaging them cancels part of each
network's individual error (about 10 % lower mean error, Section 5.2).

## 8. The same arithmetic for other situations

Wrapping steps 3-7 in one function gives a complete predictor in a dozen
lines. It agrees with the library across gaps (Overtake Mode on up to 1 s,
as the rules give it) and with Boost switched off:

In [13]:
def predict_by_hand(scenario):
    cat_index = {col: VOCABS[vocab].index(scenario[col]) for col, vocab in CATEGORICAL.items()}
    z = np.array([(scenario[c] - pre["means"][c]) / pre["stds"][c] for c in NUMERIC])
    flags = np.array([float(scenario[c]) for c in BINARY])
    return float(np.mean([forward(n, cat_index, z, flags) for n in nets]))


predictor = OvertakePredictor.load()
rows = []
for boost in (True, False):
    for gap in (0.25, 0.5, 0.75, 1.0, 1.25, 1.5):
        s = {**DEFAULTS, "gap_s": gap, "overtake_mode": gap <= 1.0, "attacker_boost": boost}
        rows.append({"Boost": "on" if boost else "off", "gap (s)": gap,
                     "by hand": predict_by_hand(s), "library": predictor.predict(s)})
check = pd.DataFrame(rows)
print(f"largest difference: {(check['by hand'] - check['library']).abs().max():.1e}")
check.round(4)

largest difference: 6.0e-08


,Boost,gap (s),by hand,library
0,on,0.25,0.8572,0.8572
1,on,0.50,0.7765,0.7765
2,on,0.75,0.6356,0.6356
3,on,1.00,0.4398,0.4398
4,on,1.25,0.0890,0.0890
5,on,1.50,0.0365,0.0365
6,off,0.25,0.1261,0.1261
7,off,0.50,0.0517,0.0517
8,off,0.75,0.0195,0.0195
9,off,1.00,0.0074,0.0074


## 9. How much arithmetic one prediction takes

Counting as in Section 6.4 of the documentation (`overtake.experiments.nn_flops`):
a multiply-add is 2 floating-point operations (FLOPs), a bias add 1, a SiLU
about 5 (exponential, add, divide, multiply). Embedding look-ups read memory
and do no arithmetic.

In [14]:
layers = [("first layer", 33, 64), ("second layer", 64, 64), ("output", 64, 1)]
count = pd.DataFrame([{"layer": name, "weights": a * b, "biases": b,
                       "FLOPs": 2 * a * b + b + (5 * b if b > 1 else 0)} for name, a, b in layers])
per_network = count["FLOPs"].sum()
weights_bytes = 4 * sum(v.size for v in net.values())  # float32: 4 bytes per number
print(f"{per_network:,} FLOPs per network, {5 * per_network:,} for the ensemble")
print(f"{weights_bytes / 1e3:.1f} kB of weights per network, {5 * weights_bytes / 1e3:.0f} kB for the ensemble")
print(f"arithmetic intensity for a single prediction: {5 * per_network / (5 * weights_bytes):.2f} FLOP/byte")
count

13,313 FLOPs per network, 66,565 for the ensemble
26.4 kB of weights per network, 132 kB for the ensemble
arithmetic intensity for a single prediction: 0.51 FLOP/byte


,layer,weights,biases,FLOPs
0,first layer,2112,64,4608
1,second layer,4096,64,8576
2,output,64,1,129


A single prediction does about half a FLOP per byte of weights it reads, so it
is limited by memory speed, not arithmetic. Predicting many scenarios at
once reuses each weight for every scenario in the batch, which is why batched
predictions reach 81 GFLOP/s (Section 6.4).

## 10. Where the numbers come from: one training step

Training showed the network 240,000 simulated attempts, each labelled 1
(overtake completed) or 0. For one attempt with predicted probability $p$
and outcome $y$, the loss is the binary cross-entropy

$$L = -\left[\, y \ln p + (1 - y) \ln(1 - p) \,\right].$$

With $p = \sigma(\text{logit})$, the chain rule gives a simple derivative,
$\partial L / \partial\,\text{logit} = p - y$: the prediction error itself. For
the output layer, where $\text{logit} = w \cdot h^{(2)} + b$,

$$\frac{\partial L}{\partial w} = (p - y)\, h^{(2)}, \qquad \frac{\partial L}{\partial b} = p - y.$$

Continuing the chain rule backwards through the second layer, the first
layer and the embeddings gives the gradient for every one of the 6,589
numbers (backpropagation). The optimiser (AdamW) then moves each number a
small step against its gradient. Repeated over batches of 1,024 attempts for
up to 100 passes through the data, this is how the weights got their values.

Below, the output-layer gradient for the default scenario with outcome
$y = 1$, by hand, checked against PyTorch's automatic differentiation:

In [15]:
y = 1.0
p0 = forward(nets[0], cat_index, z, flags)
grad_w_hand = (p0 - y) * h2
grad_b_hand = p0 - y

model = predictor.models[0]
model.eval()  # dropout off, as at prediction
cat_t, num_t = predictor.preprocessor.transform(pd.DataFrame([scenario]))
loss = torch.nn.functional.binary_cross_entropy_with_logits(model(cat_t, num_t), torch.tensor([y]))
model.zero_grad()
loss.backward()
grad_w_torch = model.mlp[6].weight.grad.numpy()[0]
grad_b_torch = model.mlp[6].bias.grad.numpy()[0]

print(f"loss = {-np.log(p0):.4f} (hand)  {loss.item():.4f} (PyTorch)")
print(f"dL/db: {grad_b_hand:.6f} (hand)  {grad_b_torch:.6f} (PyTorch)")
print(f"dL/dw, largest difference over 64 weights: {np.abs(grad_w_hand - grad_w_torch).max():.1e}")
assert np.allclose(grad_w_hand, grad_w_torch, atol=1e-5)

loss = 0.8994 (hand)  0.8994 (PyTorch)
dL/db: -0.593195 (hand)  -0.593195 (PyTorch)
dL/dw, largest difference over 64 weights: 1.3e-07


## Summary

* The network is three matrix-vector products with a SiLU between them,
  preceded by table look-ups for the categories and followed by a sigmoid:
  33 → 64 → 64 → 1. Five such networks are averaged.
* One prediction is 66,565 floating-point operations (13,313 per network)
  on 6,589 numbers per network.
* Nothing in the arithmetic is specific to F1. Everything the network
  "knows" about overtaking is in those numbers, which gradient descent set by
  making the predicted probabilities match 240,000 simulated outcomes. That
  is why the simulator's validity (Sections 5.1 and 5.4 of the documentation)
  limits the network's.